# 02.06 · Streaming agents

**Module 02 · Tools & Your First Agents** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 90–120 min |
| 📋 Prerequisites | Lessons 02.03–02.04. Lesson 01.03 (streaming a model). |
| 🔑 Keys | `GROQ_API_KEY` (optional) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Stream** an agent's progress step by step with `stream_mode="updates"`.
2. **Stream** the model's tokens with `stream_mode="messages"`, and filter them by node.
3. **Emit** and receive custom progress events (`stream_writer` + `stream_mode="custom"`).
4. **Combine** modes with `version="v2"` stream parts (`type`, `ns`, `data`) and render them with `mastery.agents.format_stream_part`.
5. **Try** the new `stream_events(version="v3")` API, and know it's marked experimental.
6. **Stream** asynchronously with `astream` in a notebook.

## 📚 Official reading (verified 2026-10-12)

| Page | Read it for |
|---|---|
| [Streaming](https://docs.langchain.com/oss/python/langchain/streaming) | **Main page.** Stream modes for agents, `version="v2"` parts, tool-call streaming, custom updates |
| [Event streaming](https://docs.langchain.com/oss/python/langchain/event-streaming) | `stream_events(version="v3")`: typed projections (`.messages`, `.tool_calls`, `.output`) |
| [LangGraph streaming](https://docs.langchain.com/oss/python/langgraph/streaming) | All stream modes (`values`, `updates`, `messages`, `custom`, `debug`) and subgraph namespaces |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-groq>=1.1.3,<2" "langgraph>=1.2.14,<2"

**You should now see** nothing, or *"Note: you may need to restart the kernel to use updated packages."* Restart only if a package was **upgraded** after you had already imported it (Lesson 00.07). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.03).

In [ ]:
from importlib import metadata

for package in ("langchain", "langchain-core", "langgraph",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/02-tools-and-agents/).
import sys
from pathlib import Path

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone.")

# Step 2: Import the course package (installed editable in Lesson 00.02; src/ fallback otherwise).
try:
    import mastery
except ModuleNotFoundError:
    sys.path.insert(0, str(REPO_ROOT / "src"))
    import mastery
    print("ℹ️ Loaded mastery from src/. Fix once: python -m pip install -e \".[dev]\"")
print("Repo root:", REPO_ROOT)

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
# Step 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

---
## Part 1 · Why stream an agent?

A model call takes ~1 s on Groq; an agent with three tool rounds takes several seconds, and a slow tool can take much longer. With `invoke`, the user stares at a spinner. Streaming lets you show **what the agent is doing** (*"calling days_between..."*), **progress** from slow tools, and the **answer as it's written**.

| `stream_mode` | You get | Use it for |
|---|---|---|
| `"updates"` | what each node added, after each step | progress UI: tool calls, tool results |
| `"messages"` | `(token chunk, metadata)` as the model generates | typing effect |
| `"custom"` | whatever tools send with `runtime.stream_writer` | progress bars, status lines |
| `"values"` | the whole state after each step | debugging (heavy) |

Pass a **list** of modes to get several at once. With `version="v2"` every item is a dict: `{"type": mode, "ns": namespace, "data": payload}`, which is the format this course uses everywhere. (Without `version="v2"`, a list of modes yields `(mode, data)` tuples, the older format you'll see in tutorials.)

In [ ]:
# 🧪 Offline: the agent we'll stream. A slow-ish tool reports progress through stream_writer.
import time

from langchain.agents import create_agent
from langchain.messages import AIMessage
from langchain.tools import ToolRuntime, tool
from mastery.agents import format_stream_part
from mastery.testing import ScriptedChatModel
from mastery.tools import calculator, days_between


@tool
def scan_statements(month: str, runtime: ToolRuntime) -> str:
    """Scan the user's bank statements for one month (YYYY-MM) and count transactions."""
    for page in (1, 2, 3):
        runtime.stream_writer({"tool": "scan_statements", "page": page, "of": 3})
        time.sleep(0.05)  # Pretend each page takes a while.
    return f"{month}: 41 transactions found"


def make_scripted():
    return ScriptedChatModel(responses=[
        AIMessage("", tool_calls=[{"name": "scan_statements", "args": {"month": "2026-09"}, "id": "s1"}]),
        AIMessage("", tool_calls=[{"name": "calculator", "args": {"expression": "41 / 30"}, "id": "s2"}]),
        "In September you made 41 transactions, about 1.37 per day.",
    ])


TOOLS = [scan_statements, calculator, days_between]
QUESTION = {"messages": [{"role": "user", "content": "How many transactions per day did I make in September 2026?"}]}
agent = create_agent(make_scripted(), tools=TOOLS)
print("ready")

---
## Part 2 · `updates`: one event per step

In [ ]:
# 🧪 Offline: stream_mode="updates" with v2 parts.
for part in agent.stream(QUESTION, stream_mode="updates", version="v2"):
    node, update = next(iter(part["data"].items()))
    last = update["messages"][-1]
    what = last.tool_calls[0]["name"] + str(last.tool_calls[0]["args"]) if getattr(last, "tool_calls", None) else last.content
    print(f"type={part['type']} ns={part['ns']} node={node:<6} → {what}")

**You should now see** five lines: `model → scan_statements{...}`, `tools → 2026-09: 41 transactions found`, `model → calculator{...}`, `tools → 1.3666...`, `model → In September...`. Each line arrives **when that step finishes**. `ns=()` means the top-level graph (subagents later in the course get their own namespace).

---
## Part 3 · `messages`: tokens as they're generated

Each `messages` part's `data` is a tuple `(chunk, metadata)`. `chunk` is an `AIMessageChunk` (or a `ToolMessage` when a tool finishes); `metadata["langgraph_node"]` says which node produced it. Tool-call arguments also stream, as `tool_call_chunks` with partial JSON.

In [ ]:
# 🧪 Offline: print text tokens from the model node only.
agent = create_agent(make_scripted(), tools=TOOLS)

for part in agent.stream(QUESTION, stream_mode="messages", version="v2"):
    chunk, metadata = part["data"]
    if metadata.get("langgraph_node") != "model":
        continue                                     # Skip ToolMessages from the tools node.
    if chunk.tool_call_chunks:
        for tc in chunk.tool_call_chunks:
            print(f"\n[tool call chunk] name={tc['name']} args={tc['args']}")
    elif chunk.text:
        print(chunk.text, end="|", flush=True)       # "|" shows where each chunk ends.
print()

**You should now see** two `[tool call chunk]` lines and then the answer split into word-sized pieces: `In |September |you |...`. A real model sends many more, smaller chunks (and splits the tool-call JSON across chunks: assemble them with `+` or wait for the `updates` event, which has the complete call).

---
## Part 4 · All three modes at once

In [ ]:
# 🧪 Offline: updates + messages + custom, rendered by mastery.agents.format_stream_part.
agent = create_agent(make_scripted(), tools=TOOLS)

for part in agent.stream(QUESTION, stream_mode=["updates", "messages", "custom"], version="v2"):
    line = format_stream_part(part)
    if line is None:
        continue
    if part["type"] == "messages":
        print(line, end="", flush=True)              # Tokens: same line.
    else:
        print(("\n" if part["type"] == "updates" else "") + line)
print()

**You should now see**, in order: `[model] 🤖 → scan_statements(month='2026-09')`, three `[progress] {"tool": "scan_statements", "page": 1, "of": 3}` lines (dicts are shown as JSON), `[tools] 🔧 ...41 transactions found`, the calculator round, then the answer tokens followed by the same answer as a `[model]` update.

That last duplication is expected: tokens arrive via `messages` *while* the model writes, and the complete message arrives via `updates` *after*. A UI typically shows tokens live and uses the update to finalize. `format_stream_part` is ~20 lines in `src/mastery/agents.py`; read it.

### What just happened?

- One `stream()` call multiplexed three channels. Each part says which one (`type`), where it came from (`ns`, empty for the top-level agent) and carries the payload (`data`).
- `custom` parts came from **inside** a tool, while it was still running, because `runtime.stream_writer` writes straight into the stream.
- Order is arrival order: you can't assume all tokens come before the update, so render each part as it arrives.

In [ ]:
# 💸 Live: the same display with the real model (3–4 short calls).
if HAVE_GROQ_KEY:
    from mastery.models import get_model

    live_agent = create_agent(get_model("fast", temperature=0, reasoning_effort="low"), tools=TOOLS,
                              system_prompt="Use the tools for every number. Answer in one sentence.")
    for part in live_agent.stream(QUESTION, {"recursion_limit": 12}, stream_mode=["updates", "messages", "custom"], version="v2"):
        line = format_stream_part(part)
        if line:
            print(line, end="", flush=True) if part["type"] == "messages" else print(("\n" if part["type"] == "updates" else "") + line)
    print()
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

**You should now see** the same structure with real tokens (many small chunks) and, if the model did what we asked, `scan_statements` then `calculator(expression='41 / 30')`. The answer tokens may arrive in a burst: Groq is fast, and reasoning tokens (not printed) come first.

---
## Part 5 · `stream_events(version="v3")`: the new API (experimental)

The [event streaming](https://docs.langchain.com/oss/python/langchain/event-streaming) page recommends a newer interface: `agent.stream_events(input, version="v3")` returns a run object with **typed projections** you iterate separately:

- `.messages`: one `ChatModelStream` per model call, with `.node`, `.text` (iterate for tokens) and `.output` (the final `AIMessage`),
- `.tool_calls`: one `ToolCallStream` per tool call (`.tool_name`, `.input`, `.output`),
- `.values`, `.output` (the final state), and `.interleave("messages", "tool_calls", ...)` to get several in arrival order.

In langgraph 1.2.14 it emits `LangChainBetaWarning: The v3 streaming protocol on Pregel is experimental.` We'll show it (it's where the API is heading) and keep `stream(..., version="v2")` as the course default until v3 is stable.

In [ ]:
# 🧪 Offline: the v3 API. Expect a LangChainBetaWarning the first time.
agent = create_agent(make_scripted(), tools=TOOLS)
run = agent.stream_events(QUESTION, version="v3")

for kind, item in run.interleave("messages", "tool_calls"):
    if kind == "tool_calls":
        print(f"🔧 {item.tool_name}({item.input})")
    else:
        text = "".join(item.text)                    # Iterating .text yields token deltas.
        print(f"🤖 [{item.node}] {text or '(tool call only)'}")

final_state = run.output
print("\nfinal answer:", final_state["messages"][-1].text)

**You should now see** `🤖 [model] (tool call only)`, `🔧 scan_statements({'month': '2026-09'})`, the calculator pair, and `🤖 [model] In September you made 41 transactions...`. Note `.text` on the final message: with v3 the content is a **list of content blocks**, so `message.content` is not a plain string here; `.text` always is.

---
## Part 6 · Async streaming

Web servers (FastAPI, Module 08) are async: use `astream` (or `astream_events`). Jupyter runs an event loop, so you can `await` and `async for` directly in a cell. (In a `.py` script, wrap it in `async def main()` and `asyncio.run(main())`.)

In [ ]:
# 🧪 Offline: async streaming, top-level `async for` in a notebook cell.
agent = create_agent(make_scripted(), tools=TOOLS)

async for part in agent.astream(QUESTION, stream_mode=["updates", "custom"], version="v2"):
    print(format_stream_part(part))

> **Async tools:** a sync tool (`def`) inside an async run is executed in a thread pool, which is fine for most tools. For I/O-heavy tools, write `async def` tools so they don't block. Never call `time.sleep` in an async tool; use `await asyncio.sleep`.

## ⚠️ Pitfalls

| Symptom | Cause | Fix |
|---|---|---|
| Items are tuples, not dicts | `version="v2"` missing | Pass `version="v2"` (course default) |
| Tool output printed as if it were model tokens | `messages` mode also emits `ToolMessage`s | Filter on `metadata["langgraph_node"] == "model"` or the chunk type |
| Answer printed twice | Tokens (`messages`) + final message (`updates`) | Show tokens live; use the update to finalize, or pick one |
| Tool-call args look like broken JSON | `tool_call_chunks` hold partial JSON | Add chunks together, or read the complete call from `updates` |
| Nothing streams until the end | Called `invoke`, or a tool blocks for a long time | `stream`/`astream`; emit progress with `stream_writer` |
| `LangChainBetaWarning` | `stream_events(version="v3")` is experimental | Expected; use v2 `stream` in production code for now |
| `SyntaxError: 'async for' outside async function` in a script | Top-level `async for` only works in notebooks | `asyncio.run(main())` |
| `.content` is a list, not a string | v3 / content blocks | Use `message.text` |

## ⚠️ Production notes

> ⚠️ **Production note: stream to users, log the final state.**
> Stream `updates` (+ `messages` for a typing effect) to the client, but log and evaluate the **final state** (all messages, tool calls, `structured_response`). Streams can be cut off by the client; the trace in LangSmith is your record.

> ⚠️ **Production note: don't stream structured output from Groq's JSON mode.**
> Groq doesn't support streaming with `response_format` JSON schema. Use `ToolStrategy` (Lesson 02.05); the `structured_response` arrives in the final `updates` event.

> ⚠️ **Production note: filter what you stream.**
> Tool results and reasoning may contain data the user shouldn't see (other users' rows, internal ids). Build your stream output from an allow-list (node names, event types), not by forwarding everything.

## 🏋️ Exercises

1. **(Easy)** Stream with `stream_mode="values"` and print `len(part["data"]["messages"])` for each part. How many parts are there, and why one more than `updates`?
2. **(Medium)** Make `scan_statements` emit `{"percent": 33}`, `66`, `100` and render a text progress bar `[####------] 66%` from the `custom` events.
3. **(Medium)** With the live agent, count how many `messages` chunks the final answer took and the time from the first to the last chunk (`time.perf_counter()`).
4. **(Stretch)** Write `async def collect(agent, question) -> dict` that streams `updates` with `astream` and returns `{"tool_calls": [...names], "answer": str}`. Run two questions concurrently with `asyncio.gather` (scripted models) and print both.

<details>
<summary><b>Show solutions</b></summary>

**1.** Six: `values` also emits the **input** state before the first step.

**4.**
```python
import asyncio

async def collect(agent, question):
    names, answer = [], ""
    async for part in agent.astream(question, stream_mode="updates", version="v2"):
        for update in part["data"].values():
            for m in update.get("messages", []):
                names += [c["name"] for c in getattr(m, "tool_calls", [])]
                if m.type == "ai" and not m.tool_calls:
                    answer = m.text
    return {"tool_calls": names, "answer": answer}

a1, a2 = create_agent(make_scripted(), tools=TOOLS), create_agent(make_scripted(), tools=TOOLS)
print(await asyncio.gather(collect(a1, QUESTION), collect(a2, QUESTION)))
```
</details>

## ✅ Checkpoint

- [ ] I can stream `updates`, `messages` and `custom` and say what each contains.
- [ ] I use `version="v2"` and read `type`, `ns`, `data`.
- [ ] I filter `messages` by `langgraph_node` and handle tool-call chunks.
- [ ] I can send progress from a tool with `runtime.stream_writer`.
- [ ] I tried `stream_events(version="v3")` and know it's experimental.
- [ ] I can stream with `astream` in a notebook and in a script.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

On the [Event streaming](https://docs.langchain.com/oss/python/langchain/event-streaming) page, find what `.extensions` and `.subagents` are for. Then check the latest [LangGraph release notes](https://github.com/langchain-ai/langgraph/releases): is the v3 protocol still marked experimental? Log what you find, with the date, in [`docs/research-log.md`](../../docs/research-log.md).

## Summary

- `stream_mode="updates"` → steps; `"messages"` → tokens; `"custom"` → your progress events.
- `version="v2"` gives uniform `{"type", "ns", "data"}` parts; combine modes in a list.
- `messages` also carries tool messages and tool-call chunks: filter and assemble.
- `stream_events(version="v3")` offers typed projections; it's experimental in langgraph 1.2.14.
- `astream` works with `async for` in notebooks; scripts need `asyncio.run`.

---
← [02-05-agent-structured-output.ipynb](02-05-agent-structured-output.ipynb) · [Module 02 README](README.md) · [02-07-studio-first-look.ipynb](02-07-studio-first-look.ipynb) →